For each NACE Class get the 100 chunks that scored highest across all the reports 

In [74]:
import pandas as pd
import glob
import os
import tqdm
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.append("..")

wor_dir = "/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis"
os.chdir(wor_dir)
#wor_dir =" "
from test_base import *

In [75]:
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/stoxx_600_overview.csv"
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"
overview_path = wor_dir + "/data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"
overview_path = wor_dir + "/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_overview.csv"

In [76]:
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4/"

raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = wor_dir + "/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3_1/"
raw_data_path = wor_dir + "/results/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = wor_dir + "/results/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = wor_dir + "/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = wor_dir + "/results/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3/"

In [77]:
reports = glob.glob(raw_data_path + "*/*_short.csv")
reports = glob.glob(raw_data_path + "*/*_long.csv")
len(reports)

3053

In [78]:
sample_ratio = 1

In [79]:
max_elements_per_class = 1000000

top_k_sentences = 200000

In [80]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
filter_only_right_chunks = True

In [81]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
with_null_classifiers = True

In [82]:
new_threshold_cos_sin = 0.4

In [83]:
nace_level_descriptions = 3
nace_level = 3
assert nace_level_descriptions >= nace_level

In [84]:
training_data_path = "data/training_data/approach_1"

In [85]:
suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "") + f"__nace_level_{nace_level}" + f"__cos_thres_{new_threshold_cos_sin}"

# "_subsample" if sample_ratio != 1 else ""
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix)
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path

'data/training_data/approach_1/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3__sample_ratio_1__filter_only_right_chunks__with_null_classifiers__nace_level_3__cos_thres_0.4'

In [86]:
#df_overview = pd.read_excel(overview_path)
df_overview = pd.read_csv(overview_path)
df_overview

,Unnamed: 0,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
0,0,CA05335P1099,Auxly Cannabis Group Inc.,1,1987.0,CAN,05335P109,19960301.0,CAN,003V9K-E,...,CA05335P1099,BDGMQB,Auxly Cannabis Group Inc.,1.0,XLY-CA,1,SHARE,BDGMQB3,A,Auxly Cannabis Group Inc.2.pdf
1,1,JP3947800003,"MEGMILK SNOW BRAND Co., Ltd.",1,2009.0,JPN,J41966102,20220727.0,JPN,0833Y1-E,...,JP3947800003,B3ZC07,"MEGMILK SNOW BRAND Co., Ltd.",1.0,MMSBF-US,0,SHARE,BKQN701,A,"MEGMILK SNOW BRAND Co., Ltd.2.pdf"
2,2,ID1000167901,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,20220527.0,IDN,@NA,...,ID1000167901,BMBMZG,PT Cilacap Samudera Fishing Industry Tbk,1.0,ASHA-ID,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf
3,3,JP3843250006,Hokuto Corporation,1,1964.0,JPN,J2224T102,19941202.0,JPN,05HY7N-E,...,JP3843250006,643271,Hokuto Corporation,1.0,1379-JP,1,SHARE,6432715,A,Hokuto Corporation1.pdf
4,4,VN000000VTQ6,Viet Trung Quang Binh Joint Stock Co,0,1961.0,VNM,Y937XS108,NaN,VNM,@NA,...,VN000000VTQ6,BMCR2W,Viet Trung Quang Binh Joint Stock Co,1.0,VTQ-VN,0,SHARE,BMCR2W8,A,Viet Trung Quang Binh Joint Stock Co3.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3274,5783,US0404432025,Aristocrat Group Corp.,0,2011.0,USA,040443202,20120313.0,USA,00DDP9-E,...,US0404432025,04044320,Aristocrat Group Corp.,1.0,ASCC-US,1,SHARE,BWX6257,S,Aristocrat Group Corp.1.pdf
3275,73228,KYG816BW1095,Sino-Life Group Limited,1,2005.0,HKG,G816BW109,20090909.0,HKG,00C60C-E,...,KYG816BW1095,B409GR,Sino-Life Group Limited,1.0,8296-HK,1,SHARE,B409GR3,S,Sino-Life Group Limited1.pdf
3276,86319,KYG9477E1070,Water Oasis Group Limited,1,1998.0,HKG,G9477E107,20070905.0,HKG,00610B-E,...,KYG9477E1070,651233,Water Oasis Group Limited,1.0,WOSSF-US,0,SHARE,B02V9Q9,S,Water Oasis Group Limited1.pdf
3277,66027,US76119X1054,"Reservoir Media, Inc.",1,2007.0,USA,76119X105,20210105.0,USA,0NVYZ2-E,...,US76119X1054,76119X10,"Reservoir Media, Inc.",1.0,RSVR-US,1,SHARE,BP0B9H7,S,"Reservoir Media, Inc.1.pdf"


In [87]:
df_nace_codes_descriptions = pd.read_csv("data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

In [88]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):

    df = pd.read_csv(report)

    if filter_only_right_chunks: 
        report_name = os.path.basename(report).replace(".txt_long.csv", "") + ".pdf"
        report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
        report_code = get_all_level(report_code)[nace_level]
        if nace_level == nace_level_descriptions: 
            filter_column = list(filter(lambda x: "Scores_"+str(report_code) in x, df.columns))
        else:
            filter_column = []
            for column in df.columns: 
                if "Scores" in column: 
                    if get_all_level(column.split("_")[1])[nace_level]==report_code: 
                        filter_column.append(column)
        df = df[["Sentences"]+filter_column]

    if nace_level == 1: 
        # filter some nace classes
        scores = df[[column for column in df.columns if ("Scores" in column) and get_all_level(column.split("_")[1], df_nace_codes_descriptions)[nace_level] in filter_level_1_classes]].columns
    else: 
        scores = df[[column for column in df.columns if ("Scores" in column)]].columns
    
    for score in scores:
        temp = df[df[score].notna()][["Sentences", score]]  
        try: 
            temp["NACE_Code"] = get_all_level(score.split("_")[1])[nace_level]
        except IndexError: 
            continue
        temp = temp.rename(columns={score: "Score"})
        result = pd.concat([result, temp])

  0%|                                                                                                                                                                                        | 0/3053 [00:00<?, ?it/s]/tmp/ipykernel_558622/32301125.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, temp])
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3053/3053 [03:08<00:00, 16.19it/s]


In [89]:
result.head()

,Sentences,Score,NACE_Code
0,nzx code principle nzx code recommendation key...,0.143800,87.2
1,lease liabilities and rightofuse assets were s...,0.232842,87.2
2,assets are assessed for impairment whenever ev...,0.282811,87.2
3,. historical governance costs. these relate to...,0.179085,87.2
4,current ingoing price for subsequent resales o...,0.105164,87.2


In [90]:
os.makedirs(end_path, exist_ok=True)

In [91]:
recordings = []

In [92]:
# for each code, store the 100 with the highest similarity score to the code

full_df = []
for code in tqdm.tqdm(set(result["NACE_Code"].to_list())): 

    #if not get_all_level(code.split("_")[1], df_nace_codes_descriptions)[1] in filter_level_1_classes: 
    # if not get_all_level(code, df_nace_codes_descriptions)[nace_level] in filter_level_1_classes: 
    #     continue

    temp = result[result["NACE_Code"] == code]
    temp = temp.drop_duplicates(subset="Sentences")
    temp = temp[temp["Sentences"].apply(len) >= 100]
    temp = temp.sort_values(by="Score", ascending=False)

    if with_null_classifiers: 
        temp.loc[temp["Score"]<new_threshold_cos_sin, "NACE_Code"] = "NO_CLASS"
        temp = temp[(temp["Score"] >= new_threshold_cos_sin) | (temp["NACE_Code"]=="NO_CLASS")]
        
        class_index = temp[temp["NACE_Code"]!="NO_CLASS"].index
        no_class_index = temp[temp["NACE_Code"]=="NO_CLASS"].index

        # print(len(temp[temp["NACE_Code"]=="NO_CLASS"]))
        # print(temp[temp["NACE_Code"]=="NO_CLASS"].index)
        # print(temp.loc[no_class_index])
        # print(code)
        # print("--")

        temp = temp.loc[list(np.random.choice(no_class_index, len(class_index)))+list(class_index)]
    else:
        temp = temp[temp["Score"] >= new_threshold_cos_sin]

    number_of_elements_per_class = min(int(sample_ratio*len(temp)), max_elements_per_class, len(temp))
    random_choice = np.random.choice(len(temp), number_of_elements_per_class, replace=False)
    temp = temp.iloc[random_choice]
    temp = temp.sort_values(by="Score", ascending=False)
    temp = temp.iloc[:top_k_sentences, :]
    temp = temp.reset_index(drop=True)
    temp["Evaluation"] = None
    temp["Notes"] = None
    temp = temp[["Evaluation", "Notes", "Sentences", "Score", "NACE_Code"]]

    recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean(), "Min. Score": temp["Score"].min(), "Max. Score": temp["Score"].max()})

    text = ""
    for i, row in temp.iterrows():
        text += f"#{i}, Score: " + str(round(row["Score"], 2)) + "\n\n" + row["Sentences"] + "\n\n"

    with open(os.path.join(end_path, code.replace("/"," ")) + ".txt", "w") as f:
        f.write(text)
    
    temp = temp.drop_duplicates(subset=["Sentences"])
    temp.to_csv(os.path.join(end_path, code.replace("/"," ")) + ".csv")

    full_df.append(temp)

full_df = pd.concat(full_df, axis=0, ignore_index=True)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 190/190 [03:22<00:00,  1.06s/it]


In [93]:
statistics = full_df.groupby("NACE_Code").agg({"Sentences": "count", "Score": "mean"})

In [94]:
full_df = pd.concat([
    full_df[full_df["NACE_Code"] == "NO_CLASS"].sample(statistics.loc[statistics.index != "NO_CLASS", "Sentences"].max()), 
    full_df[full_df["NACE_Code"] != "NO_CLASS"]
    ])

In [95]:
statistics = full_df.groupby("NACE_Code").agg({"Sentences": "count", "Score": "mean"})
statistics.to_csv(end_path + "/statistics.csv")

In [96]:
statistics

,Sentences,Score
NACE_Code,,
01.1,254,0.443262
01.2,13,0.436782
01.3,117,0.435401
01.4,52,0.432058
01.5,348,0.470621
...,...,...
93.2,705,0.446700
95.1,88,0.468308
96.0,502,0.440744


In [97]:
full_df= full_df.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df

,Evaluation,Notes,text,Score,NACE_Code
114211,None,None,all our operations work to keep land clearing ...,0.359798,NO_CLASS
1264457,None,None,the results of the group for the year ended au...,0.158787,NO_CLASS
260882,None,None,provision for affordable housing is recognised...,0.220011,NO_CLASS
23726,None,None,the board has overall responsibility for the e...,0.257808,NO_CLASS
1117062,None,None,after the abovementioned acquisitions and disp...,0.233443,NO_CLASS
...,...,...,...,...,...
1322122,None,None,the companys business is managed and categoriz...,0.400575,01.6
1322123,None,None,a large workforce and their families are house...,0.400571,01.6
1322124,None,None,disclosure of related party transactions with ...,0.400545,01.6
1322125,None,None,the hectare rate and standard cattle unit appr...,0.400543,01.6


In [98]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 81912, Test size: 27304, Validation size: 27304


In [99]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [100]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [101]:
end_path

'data/training_data/approach_1/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3__sample_ratio_1__filter_only_right_chunks__with_null_classifiers__nace_level_3__cos_thres_0.4'

In [102]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'